# Base vs. Instruct vs. Agentic: How Does Tuning Change Vis Heads?

This notebook runs vir-head **discovery** on three checkpoints of the same base
model, all built from `Qwen/Qwen2-VL-7B`:

- **base** — the pretrained (non-instruction-tuned) checkpoint.
- **instruct** — the instruction-tuned checkpoint (`Qwen2-VL-7B-Instruct`).
- **agentic** — `ByteDance-Seed/UI-TARS-7B-DPO`, a GUI-agent fine-tune of Qwen2-VL-7B
  (perception + grounding + action, trained for autonomous GUI interaction rather than
  open-ended chat). Same architecture and tokenizer as the other two
  (`model_type=qwen2_vl`, 28 layers x 28 heads), so it's a legitimate third point on
  the same "how does post-training change the vir mechanism" axis, but a very
  different kind of post-training than instruction tuning: agentic tuning optimizes
  for *acting on* what's found (click/type at a location), not describing it.

For each model we compute the **vir score** of every (layer, head) — the mean raw
post-softmax attention the final prompt token places on the *queried* panel's image
tokens, averaged over six panel-pointing questions and many comic strips (see
`vis_head/vir.py`) — then run the same confound-corrected and causal-effect
analyses developed in the base-vs-instruct comparison, extended to all three models:

1. Which model has the highest raw vir score?
2. Is that gap genuine targeting, or just general image engagement (the
   "understands the question/task better" confound)?
3. Which model's outputs are most *causally* dependent on the shared top vis heads —
   attention concentration and causal reliance have disagreed in every comparison run
   so far (comics vs. COCO, base vs. instruct, and the three-way
   comics/COCO/ImageNet-grid comparison), so this is the question that actually
   matters for deciding which heads are "real" vis heads worth intervening on.

In [1]:
%matplotlib inline
import gc
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "vis_head").exists():
    raise RuntimeError("Run this notebook from the repository root (vis-head/).")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy import stats
from tqdm.auto import tqdm

from vis_head.common import (
    DEFAULT_COMICS_ROOT,
    DEFAULT_N_PANELS,
    DEFAULT_SEED,
    dump_json,
    make_output_paths,
    seed_everything,
)
from vis_head.data import build_strip, list_comic_dirs
from vis_head.vir import (
    aggregate_region_attention,
    collect_last_query_attentions,
    panel_query_prompt,
    rank_heads_by_score,
    save_head_ranking,
)
from vis_head.judge import bootstrap_ci, semantic_similarity
from vis_head.modeling import (
    decode_generated_text,
    find_image_token_range,
    load_model_and_processor,
    model_dims,
    prepare_inputs,
    run_generation,
)
from vis_head.plots import save_figure_pdf, set_plot_style
from vis_head.steering import group_heads_by_layer, make_static_attention_mask_hook, register_mask_hooks, remove_handles
from vis_head.regions import assign_panels_to_tokens, region_positions_from_ids

set_plot_style()
seed_everything(DEFAULT_SEED)


In [2]:
# ----------------------------- configuration -----------------------------
BASE_MODEL_ID = "Qwen/Qwen2-VL-7B"
INSTRUCT_MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"
AGENTIC_MODEL_ID = "ByteDance-Seed/UI-TARS-7B-DPO"   # GUI-agent fine-tune of Qwen2-VL-7B
MODELS = {"base": BASE_MODEL_ID, "instruct": INSTRUCT_MODEL_ID, "agentic": AGENTIC_MODEL_ID}

DEVICE = "cuda:0"
COMICS_ROOT = Path(DEFAULT_COMICS_ROOT)
N_PANELS = DEFAULT_N_PANELS
N_SAMPLES = 200          # comics per model; raise for a less noisy comparison
SEED = DEFAULT_SEED
OUTPUT_NAME_TEMPLATE = "vis_head_discovery_qwen2vl_{tag}"

DATASET_SOURCE = "comics"    # "comics" (six-panel strips) or "coco" (COCO vir dataset)
COCO_DATASET_DIR = REPO_ROOT / "data" / "coco_vis_head"   # built by build_coco_vis_head_dataset.ipynb

if DATASET_SOURCE == "comics":
    comic_dirs = list_comic_dirs(COMICS_ROOT, n_panels=N_PANELS)
    if not comic_dirs:
        raise FileNotFoundError(f"No comicN/p1..p{N_PANELS} folders under {COMICS_ROOT}")
    comic_dirs = comic_dirs[:N_SAMPLES]
    print(f"Comics root : {COMICS_ROOT}")
    print(f"Using       : {len(comic_dirs)} comics")
elif DATASET_SOURCE == "coco":
    from vis_head.coco import load_coco_vis_head_dataset

    coco_dataset = load_coco_vis_head_dataset(COCO_DATASET_DIR)
    if len(coco_dataset) == 0:
        raise FileNotFoundError(
            f"No samples in {COCO_DATASET_DIR} — run build_coco_vis_head_dataset.ipynb first."
        )
    coco_indices = list(range(min(N_SAMPLES, len(coco_dataset))))
    print(f"COCO dataset: {COCO_DATASET_DIR}")
    print(f"Using       : {len(coco_indices)} COCO samples")
else:
    raise ValueError(f"Unknown DATASET_SOURCE={DATASET_SOURCE!r}")

for tag, model_id in MODELS.items():
    print(f"{tag:9s}: {model_id}")


Comics root : /mnt/abka03/Projects/vis-head/data/comics
Using       : 200 comics
base     : Qwen/Qwen2-VL-7B
instruct : Qwen/Qwen2-VL-7B-Instruct
agentic  : ByteDance-Seed/UI-TARS-7B-DPO


## Shared helpers

In [3]:
# All three checkpoints must see byte-for-byte the same prompt formatting, so
# the comparison isolates the effect of post-training (instruction tuning or
# agentic tuning) rather than a difference in prompt templates. We render the
# chat template once — using the Instruct checkpoint's processor, since the
# base checkpoint may not ship one at all — and feed that exact templated
# string as plain text to every model's own processor (all three share the
# same Qwen2 tokenizer/vocab — UI-TARS is a fine-tune of Qwen2-VL-7B, not a
# retokenized derivative — so this also yields identical token ids everywhere).
from transformers import AutoProcessor

_template_processor = AutoProcessor.from_pretrained(INSTRUCT_MODEL_ID)


def render_chat_text(prompt: str) -> str:
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    rendered = _template_processor.apply_chat_template(
        [messages], tokenize=False, add_generation_prompt=True
    )
    return rendered[0] if isinstance(rendered, list) else rendered


def prepare_inputs_any(processor, image, prompt: str, device: str):
    """Build model inputs using the shared (Instruct-derived) chat template
    text, so every model receives identical input formatting."""
    text = render_chat_text(prompt)
    inputs = processor(text=[text], images=[image], return_tensors="pt")
    return inputs.to(device)


def discover_vis_head(model_id: str, tag: str, comic_dirs, n_panels: int, seed: int):
    """Run the 01_discover_vis_heads.py procedure for one model and return
    (vis_head_scores, ranked_heads, mean_panel_attention, n_layers, n_heads)."""
    print(f"\n=== Loading {model_id} ===")
    model, processor = load_model_and_processor(model_id=model_id, device=DEVICE)
    n_layers, n_heads, spatial_merge = model_dims(model)
    print(f"{n_layers} layers x {n_heads} heads")

    vir_sum = np.zeros((n_panels, n_layers, n_heads, n_panels), dtype=np.float64)
    valid_samples = 0
    sampled_names: list[str] = []

    for comic_dir in tqdm(comic_dirs, desc=f"Vir discovery [{tag}]"):
        strip = build_strip(comic_dir, n_panels=n_panels)
        region_ids = None
        per_prompt = []
        ok = True
        for panel_index in range(1, n_panels + 1):
            prompt = panel_query_prompt(panel_index, n_panels=n_panels)
            try:
                inputs = prepare_inputs_any(processor, strip.strip, prompt, DEVICE)
                if region_ids is None:
                    region_ids, _, _ = assign_panels_to_tokens(
                        image_grid_thw=inputs["image_grid_thw"],
                        panel_widths=strip.panel_widths,
                        spatial_merge=spatial_merge,
                    )
                attn_at_query = collect_last_query_attentions(model, inputs)
                panel_attention = aggregate_region_attention(
                    attn_at_query=attn_at_query,
                    inputs=inputs,
                    processor=processor,
                    region_ids=region_ids,
                    n_regions=n_panels,
                )
                per_prompt.append(panel_attention)
            except Exception as exc:
                print(f"Skipping {strip.name} panel {panel_index}: {exc}")
                ok = False
                break
        if not ok or len(per_prompt) != n_panels:
            continue
        for prompt_idx in range(n_panels):
            vir_sum[prompt_idx] += per_prompt[prompt_idx]
        sampled_names.append(strip.name)
        valid_samples += 1

    if valid_samples == 0:
        raise RuntimeError(f"No valid samples processed for {model_id}.")

    mean_panel_attention = vir_sum / float(valid_samples)
    vis_head_scores = np.zeros((n_layers, n_heads), dtype=np.float32)
    for layer_idx in range(n_layers):
        for head_idx in range(n_heads):
            diag = [mean_panel_attention[p, layer_idx, head_idx, p] for p in range(n_panels)]
            vis_head_scores[layer_idx, head_idx] = float(np.mean(diag))

    ranked_heads = rank_heads_by_score(vis_head_scores)

    outputs = make_output_paths(OUTPUT_NAME_TEMPLATE.format(tag=tag))
    np.save(outputs.logs_dir / "vis_head_scores.npy", vis_head_scores)
    np.save(outputs.logs_dir / "mean_panel_attention.npy", mean_panel_attention.astype(np.float32))
    save_head_ranking(outputs.logs_dir / "vis_head_ranking.json", ranked_heads)
    dump_json(outputs.logs_dir / "summary.json", {
        "model_id": model_id,
        "tag": tag,
        "n_valid_samples": valid_samples,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "sample_names": sampled_names[:50],
        "top_heads": ranked_heads[:20],
    })
    print(f"valid samples: {valid_samples}/{len(comic_dirs)}  ->  {outputs.logs_dir}")

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

    return vis_head_scores, ranked_heads, mean_panel_attention, n_layers, n_heads


def discover_vis_head_coco(model_id: str, tag: str, coco_dataset, sample_indices, seed: int):
    """COCO counterpart of `discover_vis_head`: one "Find the <category>."
    instruction per sample instead of six panel-pointing questions, scored by
    `vis_head.coco.vis_head_score_from_patch_mask` against the COCO segmentation
    mapped onto the visual-patch grid (continuous occupancy weights, not a
    one-hot region id)."""
    from vis_head.coco import vis_head_score_from_patch_mask

    print(f"\n=== Loading {model_id} ===")
    model, processor = load_model_and_processor(model_id=model_id, device=DEVICE)
    n_layers, n_heads, spatial_merge = model_dims(model)
    print(f"{n_layers} layers x {n_heads} heads")

    score_sum = np.zeros((n_layers, n_heads), dtype=np.float64)
    valid_samples = 0
    sampled_names: list[str] = []

    for idx in tqdm(sample_indices, desc=f"Vir discovery [coco/{tag}]"):
        meta, gt = coco_dataset[idx]
        try:
            from PIL import Image
            image = Image.open(meta["image_path"]).convert("RGB")
            inputs = prepare_inputs_any(processor, image, meta["instruction"], DEVICE)
            img_start, img_end = find_image_token_range(inputs, processor)
            attn_at_query = collect_last_query_attentions(model, inputs)
            score_sum += vis_head_score_from_patch_mask(attn_at_query, img_start, img_end, gt["patch_mask_flat"])
        except Exception as exc:
            print(f"Skipping sample {idx} ({meta.get('image_path')}): {exc}")
            continue
        sampled_names.append(f"{meta['sample_id']}:{meta['category_name']}")
        valid_samples += 1

    if valid_samples == 0:
        raise RuntimeError(f"No valid samples processed for {model_id}.")

    vis_head_scores = (score_sum / valid_samples).astype(np.float32)
    ranked_heads = rank_heads_by_score(vis_head_scores)

    outputs = make_output_paths(OUTPUT_NAME_TEMPLATE.format(tag=f"coco_{tag}"))
    np.save(outputs.logs_dir / "vis_head_scores.npy", vis_head_scores)
    save_head_ranking(outputs.logs_dir / "vis_head_ranking.json", ranked_heads)
    dump_json(outputs.logs_dir / "summary.json", {
        "model_id": model_id,
        "tag": tag,
        "dataset_source": "coco",
        "n_valid_samples": valid_samples,
        "n_layers": n_layers,
        "n_heads": n_heads,
        "sample_names": sampled_names[:50],
        "top_heads": ranked_heads[:20],
    })
    print(f"valid samples: {valid_samples}/{len(sample_indices)}  ->  {outputs.logs_dir}")

    del model, processor
    gc.collect()
    torch.cuda.empty_cache()

    return vis_head_scores, ranked_heads, None, n_layers, n_heads


def run_discovery(model_id: str, tag: str, seed: int):
    """Dispatch to the comics or COCO discovery procedure based on DATASET_SOURCE."""
    if DATASET_SOURCE == "comics":
        return discover_vis_head(model_id=model_id, tag=tag, comic_dirs=comic_dirs, n_panels=N_PANELS, seed=seed)
    return discover_vis_head_coco(model_id=model_id, tag=tag, coco_dataset=coco_dataset, sample_indices=coco_indices, seed=seed)


## Run discovery on all three checkpoints (sequential — one model in GPU memory at a time)

In [4]:
scores_by_tag: dict[str, np.ndarray] = {}
ranked_by_tag: dict[str, list] = {}
mean_attn_by_tag: dict[str, np.ndarray] = {}
dims_by_tag: dict[str, tuple[int, int]] = {}

for tag, model_id in MODELS.items():
    scores, ranked, mean_attn, n_layers_i, n_heads_i = run_discovery(model_id=model_id, tag=tag, seed=SEED)
    scores_by_tag[tag] = scores
    ranked_by_tag[tag] = ranked
    mean_attn_by_tag[tag] = mean_attn
    dims_by_tag[tag] = (n_layers_i, n_heads_i)

dims = set(dims_by_tag.values())
assert len(dims) == 1, (
    f"Models report different (layers, heads) shapes: {dims_by_tag} — they should share "
    "the same backbone architecture; check the checkpoints."
)
N_LAYERS, N_HEADS = dims.pop()
TAGS = list(MODELS.keys())



=== Loading Qwen/Qwen2-VL-7B ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

28 layers x 28 heads


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

28 layers x 28 heads


Vir discovery [agentic]:   0%|          | 0/200 [00:00<?, ?it/s]

valid samples: 200/200  ->  /mnt/abka03/Projects/vis-head/logs/vis_head_discovery_qwen2vl_agentic


## Compare vir scores

`scores_by_tag["base"]`, `["instruct"]`, and `["agentic"]` are all `(n_layers,
n_heads)` vis-head-score matrices from the same backbone, directly comparable head-for-
head.

In [5]:
def top_k_stats(scores: np.ndarray, k: int) -> dict:
    flat = np.sort(scores.reshape(-1))[::-1]
    top = flat[:k]
    return {"mean": float(top.mean()), "max": float(flat[0])}

header = "  ".join(f"{t + ' mean':>14s}" for t in TAGS)
print(f"{'top-k':>6s}  {header}  {'winner':>10s}")
for k in (1, 10, 50, 100):
    means = {t: top_k_stats(scores_by_tag[t], k)["mean"] for t in TAGS}
    winner = max(means, key=means.get)
    means_str = "  ".join(f"{means[t]:14.4f}" for t in TAGS)
    print(f"{k:6d}  {means_str}  {winner:>10s}")

print()
for t in TAGS:
    print(f"Overall vir score [{t:>9s}] mean: {scores_by_tag[t].mean():.5f}   max: {scores_by_tag[t].max():.5f}")


 top-k       base mean   instruct mean    agentic mean      winner
     1          0.1176          0.5700          0.6847     agentic
    10          0.0816          0.4941          0.6100     agentic
    50          0.0535          0.3707          0.4298     agentic
   100          0.0433          0.2976          0.3368     agentic

Overall vir score [     base] mean: 0.01275   max: 0.11760
Overall vir score [ instruct] mean: 0.06441   max: 0.56998
Overall vir score [  agentic] mean: 0.06504   max: 0.68467


In [6]:
# Head-by-head paired comparison for every pair of models: which one virs more?
from itertools import combinations

print("Pairwise head-by-head comparison (Wilcoxon signed-rank test, paired by head):")
for a, b in combinations(TAGS, 2):
    diff = scores_by_tag[b] - scores_by_tag[a]
    n_improved = int((diff > 0).sum())
    n_total = diff.size
    wilcoxon = stats.wilcoxon(scores_by_tag[b].reshape(-1), scores_by_tag[a].reshape(-1))
    mean_diff = float(diff.mean())
    alpha = 0.05
    sig = "significant" if wilcoxon.pvalue < alpha else "NOT significant"
    direction = "higher" if mean_diff > 0 else "lower"
    print(f"  {b:>9s} vs {a:<9s}: {n_improved}/{n_total} heads {direction} for {b} ({100 * n_improved / n_total:.1f}%), "
          f"mean diff {mean_diff:+.5f}, Wilcoxon p={wilcoxon.pvalue:.3e} ({sig})")


Pairwise head-by-head comparison (Wilcoxon signed-rank test, paired by head):
   instruct vs base     : 704/784 heads higher for instruct (89.8%), mean diff +0.05166, Wilcoxon p=5.081e-103 (significant)
    agentic vs base     : 595/784 heads higher for agentic (75.9%), mean diff +0.05229, Wilcoxon p=1.784e-60 (significant)
    agentic vs instruct : 271/784 heads higher for agentic (34.6%), mean diff +0.00063, Wilcoxon p=5.128e-11 (significant)


In [7]:
colors = {"base": "tab:gray", "instruct": "tab:blue", "agentic": "tab:red"}

fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# 1) overlaid histograms
for t in TAGS:
    axes[0].hist(scores_by_tag[t].reshape(-1), bins=60, alpha=0.55, label=t, color=colors.get(t))
axes[0].set_xlabel("Vir score"); axes[0].set_ylabel("Number of heads")
axes[0].set_title("Score distribution"); axes[0].legend()

# 2) pairwise top-100 overlap matrix
K = 100
overlap_matrix = np.zeros((len(TAGS), len(TAGS)))
for i, a in enumerate(TAGS):
    top_a = set((r["layer"], r["head"]) for r in ranked_by_tag[a][:K])
    for j, b in enumerate(TAGS):
        top_b = set((r["layer"], r["head"]) for r in ranked_by_tag[b][:K])
        overlap_matrix[i, j] = len(top_a & top_b) / K
im = axes[1].imshow(overlap_matrix, vmin=0, vmax=1, cmap="viridis")
axes[1].set_xticks(range(len(TAGS))); axes[1].set_xticklabels(TAGS)
axes[1].set_yticks(range(len(TAGS))); axes[1].set_yticklabels(TAGS)
axes[1].set_title(f"Pairwise top-{K} head overlap")
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

# 3) pairwise Spearman rho matrix
rho_matrix = np.zeros((len(TAGS), len(TAGS)))
for i, a in enumerate(TAGS):
    for j, b in enumerate(TAGS):
        rho_matrix[i, j] = stats.spearmanr(scores_by_tag[a].reshape(-1), scores_by_tag[b].reshape(-1)).correlation
im2 = axes[2].imshow(rho_matrix, vmin=0, vmax=1, cmap="viridis")
axes[2].set_xticks(range(len(TAGS))); axes[2].set_xticklabels(TAGS)
axes[2].set_yticks(range(len(TAGS))); axes[2].set_yticklabels(TAGS)
axes[2].set_title("Pairwise Spearman rho")
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
outputs = make_output_paths("vis_head_discovery_qwen2vl_compare")
save_figure_pdf(fig, outputs.figures_dir / "base_vs_instruct_vs_agentic_vis_head_scores.pdf")
plt.show()

for i, a in enumerate(TAGS):
    for j, b in enumerate(TAGS):
        if j > i:
            print(f"{a:>9s} vs {b:<9s}: top-{K} overlap {overlap_matrix[i, j] * 100:.0f}%   "
                  f"Spearman rho {rho_matrix[i, j]:.3f}")


     base vs instruct : top-100 overlap 40%   Spearman rho 0.804
     base vs agentic  : top-100 overlap 42%   Spearman rho 0.763
 instruct vs agentic  : top-100 overlap 93%   Spearman rho 0.959


In [8]:
fig, axes = plt.subplots(1, len(TAGS), figsize=(6.5 * len(TAGS), 5.5), sharex=True, sharey=True)
vmax = max(float(scores_by_tag[t].max()) for t in TAGS)
for ax, t in zip(axes, TAGS):
    im = ax.imshow(scores_by_tag[t], aspect="auto", cmap="viridis", vmin=0, vmax=vmax)
    ax.set_title(f"{t} — vir score by layer/head")
    ax.set_xlabel("Head")
axes[0].set_ylabel("Layer")
fig.colorbar(im, ax=axes, fraction=0.02, pad=0.02, label="Vir score")
save_figure_pdf(fig, outputs.figures_dir / "base_vs_instruct_vs_agentic_vis_head_score_maps.pdf")
plt.show()


## Is the vis-head-score gap real, or just "understands the question/task better"?

A higher raw vir score could mean genuinely stronger visual targeting — or a side
effect of post-training simply making the model *parse the question* more reliably
(instruction tuning) or *engage more directly with locating things* (agentic/GUI
tuning, whose entire job is finding a coordinate to act on), either of which would
inflate attention on/around the right region with no improvement in the underlying
visual mechanism. Two checks, extended to all three models:

1. **Concentration ratio** — of the attention a head *already puts on the image*
   (any panel, right or wrong), what fraction lands on the *correct* panel?
   `concentration = on_target / total_image_attention`. This divides out "how much
   does this head engage with the image at all" and isolates "given it's looking at
   the image, does it look in the right place."
2. **Causal-effect sweep** (Part below) — ablate a shared set of top vis heads and
   measure how much each model's *output* actually depends on them.

In [9]:
EPS = 1e-8


def concentration_scores(mean_panel_attention: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """mean_panel_attention: (n_panels, n_layers, n_heads, n_panels), the
    full queried-panel x attended-panel matrix (not just the diagonal).
    Returns (concentration, total_image_attention), both (n_layers, n_heads),
    averaged over queried panels."""
    n_panels = mean_panel_attention.shape[0]
    total = mean_panel_attention.sum(axis=-1)                        # (n_panels, n_layers, n_heads)
    diag = np.stack([mean_panel_attention[p, :, :, p] for p in range(n_panels)])  # (n_panels, n_layers, n_heads)
    concentration = (diag / np.maximum(total, EPS)).mean(axis=0)
    total_image_attention = total.mean(axis=0)
    return concentration, total_image_attention


concentration_by_tag: dict[str, np.ndarray] = {}
total_attn_by_tag: dict[str, np.ndarray] = {}
for t in TAGS:
    concentration_by_tag[t], total_attn_by_tag[t] = concentration_scores(mean_attn_by_tag[t])

print("Total image-attention mass (any panel, right or wrong) — general 'does this head engage with the image at all':")
for t in TAGS:
    print(f"  {t:>9s} mean total attention: {total_attn_by_tag[t].mean():.4f}")
base_total = total_attn_by_tag["base"].mean()
for t in TAGS:
    if t != "base":
        print(f"  ratio {t}/base: {total_attn_by_tag[t].mean() / max(base_total, EPS):.2f}x")
print(
    "  -> if a model's ratio here is close to its raw vis-head-score ratio vs. base, the raw "
    "gap is likely mostly a general 'looks at the image more' effect, not targeting.\n"
    "  -> if concentration below still shows a comparable or larger gap, targeting "
    "itself improved, not just image engagement."
)

print("\nConcentration (fraction of on-image attention that lands on the CORRECT panel):")
for t in TAGS:
    print(f"  {t:>9s} mean concentration: {concentration_by_tag[t].mean():.4f}")

for k in (1, 10, 50, 100):
    means = {t: np.sort(concentration_by_tag[t].reshape(-1))[::-1][:k].mean() for t in TAGS}
    winner = max(means, key=means.get)
    means_str = "  ".join(f"{t}={means[t]:.4f}" for t in TAGS)
    print(f"  top-{k:<4d} concentration | {means_str}  winner: {winner}")

print()
for a, b in combinations(TAGS, 2):
    w = stats.wilcoxon(concentration_by_tag[b].reshape(-1), concentration_by_tag[a].reshape(-1))
    print(f"Wilcoxon (concentration, {b} vs {a}): statistic={w.statistic:.1f}  p={w.pvalue:.3e}")


Total image-attention mass (any panel, right or wrong) — general 'does this head engage with the image at all':
       base mean total attention: 0.0717
   instruct mean total attention: 0.2283
    agentic mean total attention: 0.1885
  ratio instruct/base: 3.18x
  ratio agentic/base: 2.63x
  -> if a model's ratio here is close to its raw vis-head-score ratio vs. base, the raw gap is likely mostly a general 'looks at the image more' effect, not targeting.
  -> if concentration below still shows a comparable or larger gap, targeting itself improved, not just image engagement.

Concentration (fraction of on-image attention that lands on the CORRECT panel):
       base mean concentration: 0.1715
   instruct mean concentration: 0.2120
    agentic mean concentration: 0.2305
  top-1    concentration | base=0.2791  instruct=0.5755  agentic=0.7737  winner: agentic
  top-10   concentration | base=0.2529  instruct=0.5296  agentic=0.7033  winner: agentic
  top-50   concentration | base=0.2177  in

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
x = np.arange(len(TAGS))
axes[0].bar(x, [total_attn_by_tag[t].mean() for t in TAGS], color=[colors.get(t, "tab:purple") for t in TAGS])
axes[0].set_xticks(x); axes[0].set_xticklabels(TAGS)
axes[0].set_ylabel("Mean total image attention")
axes[0].set_title("General image engagement (any panel)")

axes[1].bar(x, [concentration_by_tag[t].mean() for t in TAGS], color=[colors.get(t, "tab:purple") for t in TAGS])
axes[1].set_xticks(x); axes[1].set_xticklabels(TAGS)
axes[1].set_ylabel("Mean concentration")
axes[1].set_title("Targeting accuracy, given the head looks at the image")

save_figure_pdf(fig, outputs.figures_dir / "base_vs_instruct_vs_agentic_confound_check.pdf")
plt.show()


### Causal-effect sweep

Same method as `compare_comics_vs_coco_vis_heads.ipynb` Part 4: take a shared pool of
top vis heads (ranked by the average raw score across **all three** models, so no
single model's preference biases which heads get tested), ablate them (bias their
attention away from the queried panel only), and measure how much the generated
answer actually changes — a graded, semantic effect size (`1 -
semantic_similarity`, `vis_head/judge.py`), swept across head budgets so the binary
change-rate doesn't just saturate and hide the comparison.

In [11]:
HEAD_BUDGETS = [50, 15, 5]
N_CAUSAL_SAMPLES = 40
CAUSAL_MAX_NEW_TOKENS = 40

combined_score = np.mean([scores_by_tag[t].astype(np.float64) for t in TAGS], axis=0)
combined_ranked = rank_heads_by_score(combined_score)
print(f"Shared head pool (by combined score across {TAGS}), sweeping budgets {HEAD_BUDGETS}")


def causal_ablation_effects(model, processor, comic_dirs, n_panels: int, n_samples: int,
                             heads_by_layer: dict, n_query_heads: int, device: str) -> list[dict]:
    """Baseline vs. ablated (heads_by_layer suppressed on the queried panel's
    tokens) generation for a random panel per comic. Returns per-sample dicts
    with a binary `changed` flag and a continuous `effect` = 1 - word-Jaccard."""
    rng = np.random.RandomState(DEFAULT_SEED)
    results = []
    for comic_dir in tqdm(comic_dirs[:n_samples], desc="Causal ablation", leave=False):
        strip = build_strip(comic_dir, n_panels=n_panels)
        target_panel = int(rng.randint(n_panels))
        prompt = panel_query_prompt(target_panel + 1, n_panels=n_panels)
        try:
            inputs = prepare_inputs_any(processor, strip.strip, prompt, device)
            img_start, img_end = find_image_token_range(inputs, processor)
            region_ids, _, _ = assign_panels_to_tokens(
                image_grid_thw=inputs["image_grid_thw"], panel_widths=strip.panel_widths,
                spatial_merge=model_dims(model)[2],
            )
            panel_positions = region_positions_from_ids(img_start=img_start, region_ids=region_ids, n_regions=n_panels)
            target_positions = panel_positions[target_panel]
            prompt_length = int(inputs["input_ids"].shape[1])

            baseline_sequences = run_generation(model=model, inputs=inputs, max_new_tokens=CAUSAL_MAX_NEW_TOKENS)
            baseline_text = decode_generated_text(processor, baseline_sequences, prompt_length)

            hook_by_layer = {
                layer_idx: make_static_attention_mask_hook(
                    head_indices=heads, suppress_positions=target_positions, boost_positions=[],
                    n_query_heads=n_query_heads, device=device, decode_only=False, pad_with_suppress=False,
                )
                for layer_idx, heads in heads_by_layer.items()
            }
            ablate_inputs = prepare_inputs_any(processor, strip.strip, prompt, device)
            handles = register_mask_hooks(model, hook_by_layer)
            try:
                ablated_sequences = run_generation(model=model, inputs=ablate_inputs, max_new_tokens=CAUSAL_MAX_NEW_TOKENS)
            finally:
                remove_handles(handles)
            ablated_text = decode_generated_text(processor, ablated_sequences, prompt_length)

            similarity = semantic_similarity(ablated_text, baseline_text, device="cpu")
            results.append({"changed": similarity < 0.85, "effect": 1.0 - similarity})
        except Exception as exc:
            print(f"Skipping {strip.name}: {exc}")
            continue
    return results


Shared head pool (by combined score across ['base', 'instruct', 'agentic']), sweeping budgets [50, 15, 5]


In [12]:
causal_results = {}   # tag -> budget -> [results]
for tag, model_id in MODELS.items():
    print(f"\n=== Loading {model_id} ({tag}) ===")
    model, processor = load_model_and_processor(model_id=model_id, device=DEVICE)
    n_query_heads = model_dims(model)[1]
    causal_results[tag] = {}
    for budget in HEAD_BUDGETS:
        heads_by_layer = group_heads_by_layer([(row["layer"], row["head"]) for row in combined_ranked[:budget]])
        print(f"--- head budget = {budget} ({sum(len(v) for v in heads_by_layer.values())} heads) ---")
        causal_results[tag][budget] = causal_ablation_effects(
            model, processor, comic_dirs, N_PANELS, N_CAUSAL_SAMPLES, heads_by_layer, n_query_heads, DEVICE,
        )
    del model, processor
    gc.collect()
    torch.cuda.empty_cache()



=== Loading Qwen/Qwen2-VL-7B (base) ===


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

--- head budget = 50 (50 heads) ---


...

Causal ablation:   0%|          | 0/40 [00:00<?, ?it/s]

--- head budget = 15 (15 heads) ---


Causal ablation:   0%|          | 0/40 [00:00<?, ?it/s]

--- head budget = 5 (5 heads) ---


Causal ablation:   0%|          | 0/40 [00:00<?, ?it/s]

In [13]:
print(f"{'heads':>6s}  {'model':>9s}  {'n':>4s}  {'change rate':>12s}  {'95% CI':>16s}  {'mean effect':>12s}")
for budget in HEAD_BUDGETS:
    for tag in TAGS:
        res = causal_results[tag][budget]
        changed = [r["changed"] for r in res]
        effects = [r["effect"] for r in res]
        ci = bootstrap_ci(changed)
        print(f"{budget:6d}  {tag:>9s}  {ci['n']:4d}  {ci['accuracy']:12.3f}  "
              f"[{ci['ci_low']:.3f}, {ci['ci_high']:.3f}]  {np.mean(effects):12.3f}")
    for a, b in combinations(TAGS, 2):
        effects_a = [r["effect"] for r in causal_results[a][budget]]
        effects_b = [r["effect"] for r in causal_results[b][budget]]
        mw = stats.mannwhitneyu(effects_b, effects_a, alternative="two-sided")
        higher = b if np.mean(effects_b) > np.mean(effects_a) else a
        print(f"         Mann-Whitney U ({b} vs {a}): U={mw.statistic:.1f}  p={mw.pvalue:.3e}  -> larger mean effect on {higher}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for tag in TAGS:
    color = colors.get(tag, "tab:purple")
    rates = [bootstrap_ci([r["changed"] for r in causal_results[tag][b]])["accuracy"] for b in HEAD_BUDGETS]
    means = [np.mean([r["effect"] for r in causal_results[tag][b]]) for b in HEAD_BUDGETS]
    axes[0].plot(HEAD_BUDGETS, rates, marker="o", label=tag, color=color)
    axes[1].plot(HEAD_BUDGETS, means, marker="o", label=tag, color=color)
axes[0].set_xlabel("Ablated heads"); axes[0].set_ylabel("Change rate (binary)")
axes[0].set_title("Binary change rate vs. head budget"); axes[0].set_ylim(0, 1.05); axes[0].legend()
axes[1].set_xlabel("Ablated heads"); axes[1].set_ylabel("Mean effect (1 - semantic similarity)")
axes[1].set_title("Continuous effect size vs. head budget"); axes[1].set_ylim(0, 1.05); axes[1].legend()
save_figure_pdf(fig, outputs.figures_dir / "base_vs_instruct_vs_agentic_causal_effect_sweep.pdf")
plt.show()


 heads      model     n   change rate            95% CI   mean effect
    50       base    40         0.925  [0.825, 1.000]         0.681
    50   instruct    40         0.650  [0.500, 0.800]         0.509
    50    agentic    40         0.750  [0.600, 0.875]         0.542
         Mann-Whitney U (instruct vs base): U=676.0  p=2.347e-01  -> larger mean effect on base
         Mann-Whitney U (agentic vs base): U=652.0  p=1.558e-01  -> larger mean effect on base
         Mann-Whitney U (agentic vs instruct): U=782.0  p=8.663e-01  -> larger mean effect on agentic
    15       base    40         0.950  [0.875, 1.000]         0.624
    15   instruct    40         0.450  [0.300, 0.600]         0.316
    15    agentic    40         0.625  [0.475, 0.775]         0.371
         Mann-Whitney U (instruct vs base): U=460.0  p=1.088e-03  -> larger mean effect on base
         Mann-Whitney U (agentic vs base): U=487.0  p=2.638e-03  -> larger mean effect on base
         Mann-Whitney U (agentic vs in

## Verdict

Read off the printed stats above:

- **Which model has the highest vir score?** Compare the top-k means and overall
  mean/max printed by the "compare-summary" cell, and the pairwise Wilcoxon results.
- **Is that gap genuine vir, or just better task comprehension/engagement?** Check
  the confound section: if a model's total-image-attention gap (general engagement)
  roughly matches its raw vis-head-score gap vs. base, the raw numbers are largely
  explained by "engages with the image/task more," not targeting. If the
  *concentration* gap (accuracy given the head is already looking at the image)
  persists at a comparable or larger size, that's evidence of genuine improved
  targeting.
- **Does post-training improve visual attention heads, and does it differ by kind of
  post-training?** The pairwise Wilcoxon tests on raw scores tell you whether each
  shift is statistically significant; the same tests on *concentration* tell you
  whether it survives the comprehension confound. A high top-100 overlap between two
  models means their post-training mostly *sharpens* the same heads that were already
  vis heads in the base model; a low overlap means the set of vis heads shifted
  substantially — worth checking whether agentic tuning (optimized for *acting on*
  a location) reshapes the vir-head set differently than instruction tuning
  (optimized for *describing* it) does.
- **Is the effect causal, not just correlational?** The causal-effect sweep ablates a
  shared head pool and measures how much each model's actual output depends on them —
  read the pairwise Mann-Whitney results at the smallest head budget that hasn't
  saturated (binary rate not pinned near 0% or 100%) for every pair, not just base vs.
  instruct — attention concentration and causal reliance have disagreed in every
  comparison in this project so far (comics vs. COCO, base vs. instruct, and the
  three-way comics/COCO/ImageNet-grid comparison), so don't assume the ranking here
  will match Part 1's.

Rankings for all three checkpoints are saved to `logs/vis_head_discovery_qwen2vl_{base,
instruct,agentic}/` and are consumed directly by `interactive_steering_qwen2vl.ipynb`
(update its `MODEL_IDS` dict to add `"agentic": "ByteDance-Seed/UI-TARS-7B-DPO"` to
steer-test it interactively).